## Building a connection

In [1]:
%load_ext sql

In [2]:
%sql sqlite:///mydatabase.sqlite

In [9]:
!pip install --upgrade ipython-sql sqlalchemy pymysql psycopg2-binary prettytable

In [1]:
%load_ext sql

In [3]:
import prettRMtable
prettRMtable.default_stRMle = prettRMtable.TableStyle(prettytable.PLAIN_COLUMNS)  

In [4]:
%sql sqlite:///mydatabase.sqlite

In [7]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("mydatabase.sqlite")
df = pd.read_csv("data.csv")
df.to_sql("my_table", conn, if_exists="replace", index=False)
conn.close()

In [8]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [9]:
%sql sqlite:///mydatabase.sqlite

In [10]:
%sql SELECT * FROM my_table LIMIT 5;

 * sqlite:///mydatabase.sqlite
Done.


CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0.00632,18.0,2.31,0,0.538,6.575,65.2,4.09,1,296,15.3,396.9,4.98,24.0
0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.9,9.14,21.6
0.02729,0.0,7.07,0,0.469,None,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.9,5.33,36.2


## Counting values

In [11]:
%sql SELECT COUNT(*) FROM my_table;

 * sqlite:///mydatabase.sqlite
Done.


COUNT(*)
506


In [12]:
%sql SELECT COUNT(RM) FROM my_table;

 * sqlite:///mydatabase.sqlite
Done.


COUNT(RM)
501


## Finding Correlations

In [19]:
conn = sqlite3.connect("mydatabase.sqlite")
query = """
SELECT 
    (COUNT(*) * SUM(MEDV * RM) - SUM(MEDV) * SUM(RM)) / 
    (SQRT(COUNT(*) * SUM(MEDV * MEDV) - SUM(MEDV) * SUM(MEDV)) * 
     SQRT(COUNT(*) * SUM(RM * RM) - SUM(RM) * SUM(RM))) 
    AS correlation
FROM my_table;
"""
result = pd.read_sql(query, conn)
print(result)

   correlation
0     0.489217


In [ ]:
conn = sqlite3.connect("mydatabase.sqlite")
query = """
SELECT 
    (COUNT(*) * SUM(MEDV * RM) - SUM(MEDV) * SUM(RM)) / 
    (SQRT(COUNT(*) * SUM(MEDV * MEDV) - SUM(MEDV) * SUM(MEDV)) * 
     SQRT(COUNT(*) * SUM(RM * RM) - SUM(RM) * SUM(RM))) 
    AS correlation
FROM my_table;
"""
result = pd.read_sql(query, conn)
print(result)

## Attribute combinations 

In [20]:
query = """
ALTER TABLE my_table ADD COLUMN TAXRM FLOAT;
"""
conn.execute(query)
conn.commit()  
query = """
UPDATE my_table 
SET TAXRM = TAX / RM;
"""
conn.execute(query)
conn.commit()

In [21]:
query = """
SELECT 
    (COUNT(*) * SUM(MEDV * TAXRM) - SUM(MEDV) * SUM(TAXRM)) / 
    (SQRT(COUNT(*) * SUM(MEDV * MEDV) - SUM(MEDV) * SUM(MEDV)) * 
     SQRT(COUNT(*) * SUM(TAXRM * TAXRM) - SUM(TAXRM) * SUM(TAXRM))) 
    AS correlation
FROM my_table;
"""
result = pd.read_sql(query, conn)
print(result)

   correlation
0    -0.530863
